# K-Medoids Clustering with Multiple Time Series Distance Methods

This notebook provides a comprehensive comparison of various time series distance methods using K-Medoids clustering on UCR datasets.

## How to Use

### Step 1: Run All Cells in Order
Execute all cells to define all functions and distance methods.

### Step 2: Select Your Method
In **Cell 9 (Main Execution)**, change the `ALG` variable:
```python
ALG = "OTSW"  # Options: "OTSW", "OPW", "TCOT", "POW", "ASW", "TAOT"
```

### Step 3: Run the Experiment
Execute Cell 9 to:
- Load UCR datasets
- Compute distance matrices
- Run K-Medoids clustering
- Save results to `kmedoids_results_{method}.csv`

## Output
Results include:
- **ACC**: Clustering accuracy (after label mapping)
- **NMI**: Normalized Mutual Information
- **Time_Dist_Total**: Time to compute distance matrix
- **Time_Clust**: Time for clustering
- **Time_Tree**: Tree construction time (OTSW only)

In [1]:
# Fix NumPy compatibility - RESTART KERNEL after running this cell!
# Install NumPy 1.26.4 (last 1.x version with Python 3.12 support)
!pip uninstall -y numpy
!pip install "numpy==1.26.4"
!pip install "scipy>=1.10,<2.0"
!pip install "scikit-learn>=1.1,<1.6"
!pip install "scikit-learn-extra>=0.3.0"
!pip install tslearn matplotlib
!pip install "POT>=0.9.3"

print("\n⚠️ IMPORTANT: Please RESTART THE KERNEL before running other cells!")

Found existing installation: numpy 1.26.4
Uninstalling numpy-1.26.4:
  Successfully uninstalled numpy-1.26.4
  Using cached numpy-1.26.4-cp312-cp312-win_amd64.whl.metadata (61 kB)
Using cached numpy-1.26.4-cp312-cp312-win_amd64.whl (15.5 MB)


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opencv-python 4.12.0.88 requires numpy<2.3.0,>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.

[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip



⚠️ IMPORTANT: Please RESTART THE KERNEL before running other cells!



[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 1. OTSW Implementation (Ragged-Friendly)


In [2]:
# otsw_api.py  — RAGGED-FRIENDLY
import numpy as np
from dataclasses import dataclass
from typing import List, Tuple, Optional, Dict, Sequence, Union
import heapq

# (Optional) Use SciPy to accelerate SpMM; works without it as well
try:
    import scipy.sparse as sp
    _HAS_SCIPY = True
except Exception:
    _HAS_SCIPY = False

BIG = 1e12

# =========================
# 0) HELPERS (ragged / dense)
# =========================
def _as_ragged_list(M: Union[np.ndarray, Sequence[np.ndarray]]) -> Tuple[List[np.ndarray], int]:
    """
    Normalize input to a list of arrays (n_i, d).
    Returns (list_seq, d)
    """
    if isinstance(M, np.ndarray):
        if M.ndim != 3:
            raise ValueError("If ndarray, expect shape (m, n, d).")
        m, n, d = M.shape
        seqs = [M[i] for i in range(m)]
        return seqs, d
    # list/tuple các chuỗi (n_i, d)
    seqs = []
    d = None
    for i, xi in enumerate(M):
        xi = np.asarray(xi, dtype=float)
        if xi.ndim != 2:
            raise ValueError(f"Sequence {i} must have shape (n_i, d).")
        if d is None:
            d = xi.shape[1]
        elif xi.shape[1] != d:
            raise ValueError("All sequences must have the same feature dimension d.")
        seqs.append(xi)
    if d is None:
        raise ValueError("Empty sequence list.")
    return seqs, d

def _linearize_points_ragged(M: Union[np.ndarray, Sequence[np.ndarray]]):
    """
    Ragged support: 
      - P: (N, d) concatenated points
      - Sidx: (N,) series id
      - Tpos: (N,) normalized time in [0,1) for each point (i / n_i)
      - lengths: (m_seq,) length of each series
      - d: number of channels
    """
    seqs, d = _as_ragged_list(M)
    m_seq = len(seqs)
    lengths = np.array([xi.shape[0] for xi in seqs], dtype=int)
    # concatenate points
    P = np.vstack(seqs) if m_seq > 0 else np.zeros((0, d))
    # series id
    Sidx = np.repeat(np.arange(m_seq, dtype=int), lengths)
    # normalized time position (avoid endpoint=1 to prevent collision at 1.0)
    Tpos_list = [ (np.arange(n_i, dtype=float) / max(n_i,1)) for n_i in lengths ]
    Tpos = np.concatenate(Tpos_list) if m_seq > 0 else np.zeros((0,), float)
    return P, Sidx, Tpos, m_seq, lengths, d

def _pairwise_sqdist(A: np.ndarray, B: np.ndarray) -> np.ndarray:
    """
    'Hybrid' distance:
      Euclid^2 on features (except last column)  +  |orderA - orderB| (last column)
    """
    assert A.ndim == 2 and B.ndim == 2, "Expect 2D arrays"
    assert A.shape[1] == B.shape[1], "Dim mismatch"
    D = A.shape[1]
    if D == 0:
        return np.zeros((A.shape[0], B.shape[0]), dtype=float)
    if D == 1:
        a_ord = A[:, 0]; b_ord = B[:, 0]
        return np.abs(a_ord[:, None] - b_ord[None, :])
    Af = A[:, :-1]; Bf = B[:, :-1]
    aa = (Af * Af).sum(1)[:, None]
    bb = (Bf * Bf).sum(1)[None, :]
    Dsq = np.clip(aa + bb - 2 * (Af @ Bf.T), 0.0, None)
    a_ord = A[:, -1]; b_ord = B[:, -1]
    Pen = np.abs(a_ord[:, None] - b_ord[None, :])
    return Dsq + Pen

@dataclass
class _Node:
    idx: np.ndarray
    height: float
    left: Optional[int]
    right: Optional[int]
    parent: Optional[int]
    is_leaf: bool

@dataclass
class OTSWModel:
    # shared runtime fields
    P: np.ndarray                  # (N, d_aug) nếu TamLe, hoặc (N, d) nếu Banded
    Sidx: np.ndarray               # (N,) id chuỗi
    Tpos: np.ndarray               # (N,) thời gian chuẩn hoá (0..1)
    lengths: np.ndarray            # (m_seq,)
    m_seq: int
    d: int                         # số kênh gốc (chưa augment)
    nodes: List[_Node]
    leaf_ids: List[int]
    leaf_index_map: Dict[int, int]
    edges: List[Tuple[int, int, float]]     # (parent, child, w_e)
    S_edge_leaf: object                     # (E, L) dense hoặc sp.csr_matrix
    centroids: np.ndarray                   # (num_nodes, D_aug)
    # meta
    mode: str                               # "tamle" | "banded"
    lam_time: float = 0.0
    lam_idx: float = 0.0
    W: float = 0.0                           # band width (tỉ lệ 0..1 cho ragged)
    # caches
    point_leaf: Optional[np.ndarray] = None  # (N,)
    H: Optional[np.ndarray] = None           # (L, m_seq)
    M: Optional[np.ndarray] = None           # (E, m_seq)
    w: Optional[np.ndarray] = None           # (E,)

# =========================
# 1) BOX TREE + GONZALEZ
# =========================
class _KDBoxTree:
    def __init__(self, leaf_size=64, max_depth=24):
        self.leaf_size = leaf_size
        self.max_depth = max_depth
        self.boxes = []
        self.X = None

    def _build(self, idx, depth):
        X = self.X[idx]
        c = X.mean(axis=0)
        r = float(np.sqrt(((X - c) ** 2).sum(1).max())) if X.shape[0] else 0.0
        bid = len(self.boxes)
        self.boxes.append({"idx": idx, "c": c, "r": r, "L": None, "R": None, "leaf": False})
        if idx.size <= self.leaf_size or depth >= self.max_depth or r == 0.0:
            self.boxes[bid]["leaf"] = True
            return bid
        var = X.var(axis=0)
        d = int(np.argmax(var))
        med = np.median(X[:, d])
        mask = X[:, d] <= med
        if mask.all() or (~mask).all():
            mid = idx.size // 2
            Lidx = idx[:mid]; Ridx = idx[mid:]
        else:
            Lidx = idx[mask]; Ridx = idx[~mask]
        L = self._build(Lidx, depth + 1)
        R = self._build(Ridx, depth + 1)
        self.boxes[bid]["L"] = L; self.boxes[bid]["R"] = R
        return bid

    def fit(self, X):
        self.X = X
        self.boxes = []
        self._build(np.arange(X.shape[0]), 0)

def _bounds_box(box, centers):
    if centers.size == 0: return 0.0, float("inf")
    d = np.sqrt(((centers - box["c"][None, :]) ** 2).sum(1))
    dmin = float(d.min())
    r = box["r"]
    return max(0.0, dmin - r), dmin + r

def _farthest_point_by_boxes(X, centers, kdt: _KDBoxTree, gap_tol=1e-6):
    if centers.size == 0: return 0, 0.0
    heap = []
    L0, U0 = _bounds_box(kdt.boxes[0], centers)
    heapq.heappush(heap, (-U0, 0, L0))
    best_idx, best_val = None, -1.0
    while heap:
        negU, bid, Lb = heapq.heappop(heap)
        Ub = -negU
        L2, U2 = _bounds_box(kdt.boxes[bid], centers)
        if U2 < Ub - 1e-12 or L2 > Lb + 1e-12:
            heapq.heappush(heap, (-U2, bid, L2)); continue
        if best_val >= U2 - 1e-15: break
        box = kdt.boxes[bid]
        if box["leaf"] or (U2 - L2) <= gap_tol:
            pts = kdt.X[box["idx"]]
            D = _pairwise_sqdist(pts, centers)  # KHÔNG sqrt
            dmin = D.min(axis=1)
            imax = int(np.argmax(dmin)); val = float(dmin[imax])
            if val > best_val: best_val, best_idx = val, int(box["idx"][imax])
            continue
        for child in (box["L"], box["R"]):
            Lc, Uc = _bounds_box(kdt.boxes[child], centers)
            heapq.heappush(heap, (-Uc, child, Lc))
    return best_idx, best_val

def _gonzalez_box_nlogk(X: np.ndarray, k: int, seed: int,
                        box_leaf_size=64, box_max_depth=24, gap_tol=1e-6):
    rng = np.random.default_rng(seed)
    n = X.shape[0]; assert 1 <= k <= n
    kdt = _KDBoxTree(leaf_size=box_leaf_size, max_depth=box_max_depth); kdt.fit(X)
    i0 = int(rng.integers(0, n)); centers = X[i0:i0+1]; C = [i0]
    for _ in range(1, k):
        idx, _ = _farthest_point_by_boxes(X, centers, kdt, gap_tol)
        C.append(idx); centers = X[np.array(C)]
    return np.array(C, dtype=int)

# =========================
# 1.1) ROUTING & PRECOMPUTE
# =========================
def _route_all_points_vectorized(model: OTSWModel) -> np.ndarray:
    N = model.P.shape[0]
    leaf_of_point = np.empty(N, dtype=np.int32)
    stack = [(0, np.arange(N, dtype=np.int32))]
    nodes = model.nodes; C = model.centroids; P = model.P
    while stack:
        nid, idxs = stack.pop()
        nd = nodes[nid]
        if nd.is_leaf:
            j = model.leaf_index_map[nid]; leaf_of_point[idxs] = j; continue
        L = nd.left; R = nd.right
        X = P[idxs]
        dl = np.linalg.norm(X - C[L], axis=1)
        dr = np.linalg.norm(X - C[R], axis=1)
        go_left = dl <= dr
        if go_left.any():    stack.append((L, idxs[go_left]))
        if (~go_left).any(): stack.append((R, idxs[~go_left]))
    return leaf_of_point

def _precompute_H_M(model: OTSWModel):
    m_seq = model.m_seq
    N = model.P.shape[0]
    L = len(model.leaf_ids)
    E = len(model.edges)
    # 1) route tất cả điểm -> lá
    point_leaf = _route_all_points_vectorized(model)  # (N,)
    model.point_leaf = point_leaf
    # 2) H (L, m_seq) — histogram mỗi chuỗi
    H = np.zeros((L, m_seq), dtype=np.float32)
    for s in range(m_seq):
        mask = (model.Sidx == s)
        if not np.any(mask): continue
        counts = np.bincount(point_leaf[mask], minlength=L).astype(np.float32)
        tot = counts.sum()
        if tot > 0: counts /= tot
        H[:, s] = counts
    model.H = H
    # 3) S_edge_leaf -> CSR (nếu có SciPy) và M = S @ H
    if _HAS_SCIPY:
        SpS = sp.csr_matrix(model.S_edge_leaf)
        model.S_edge_leaf = SpS
        M = (SpS @ H).astype(np.float32)  # (E, m_seq)
    else:
        M = (model.S_edge_leaf @ H).astype(np.float32)
    model.M = M
    # 4) Trọng số cạnh
    model.w = np.array([we for _, _, we in model.edges], dtype=np.float32)

# =========================
# 1.2) OTSW — TAM LE (ragged OK)
# =========================
def _augment_points(seq: np.ndarray, lam_time: float) -> np.ndarray:
    n = seq.shape[0]
    t = (np.arange(n, dtype=float) / max(n, 1))[:, None] * np.sqrt(lam_time)
    return np.hstack([seq, t])

def build_otsw_tamle(
    M: Union[np.ndarray, Sequence[np.ndarray]],
    lam_time: float = 5.0,
    leaf_size: int = 16,
    max_depth: int = 20,
    seed: int = 0,
    k_split: int = 2,
    box_leaf_size: int = 64,
    box_max_depth: int = 24,
) -> OTSWModel:
    """
    Xây cây global theo TamLe (augment theo thời gian chuẩn hoá → ragged friendly).
    """
    P_raw, Sidx, Tpos, m_seq, lengths, d = _linearize_points_ragged(M)
    # augment từng chuỗi rồi ghép
    P_aug_list = []
    start = 0
    for s in range(m_seq):
        n_i = lengths[s]
        seq = P_raw[start:start+n_i]
        P_aug_list.append(_augment_points(seq, lam_time))
        start += n_i
    P_aug = np.vstack(P_aug_list) if P_aug_list else np.zeros((0, d+1))
    # build tree
    nodes: List[_Node] = []; leaf_ids: List[int] = []
    def _euclid_radius(X):
        if X.shape[0] <= 1: return 0.0
        if X.shape[0] > 1024:
            I = np.random.default_rng(0).choice(X.shape[0], 1024, replace=False); Y = X[I]
        else: Y = X
        j0 = 0; d0 = np.linalg.norm(Y - Y[j0], axis=1); j1 = int(np.argmax(d0))
        d1 = np.linalg.norm(Y - Y[j1], axis=1); return 0.5 * float(d1.max())
    def build(idx: np.ndarray, depth: int, parent: Optional[int], seed_: int) -> int:
        Xsub = P_aug[idx]; h = _euclid_radius(Xsub)
        nid = len(nodes); nodes.append(_Node(idx, h, None, None, parent, False))
        if idx.size <= leaf_size or depth >= max_depth or h == 0.0:
            nodes[nid].is_leaf = True; leaf_ids.append(nid); return nid
        C = _gonzalez_box_nlogk(Xsub, k=k_split, seed=seed_,
                                box_leaf_size=box_leaf_size, box_max_depth=box_max_depth)
        centers = Xsub[C]
        lab = np.argmin(_pairwise_sqdist(Xsub, centers), axis=1)
        if k_split == 2:
            left_idx = idx[lab == 0]; right_idx = idx[lab != 0]
        else:
            cnt = np.bincount(lab, minlength=k_split); main = int(np.argmax(cnt))
            left_idx = idx[lab == main]; right_idx = idx[lab != main]
        if left_idx.size == 0 or right_idx.size == 0:
            mid = idx.size // 2; left_idx = idx[:mid]; right_idx = idx[mid:]
        L = build(left_idx, depth+1, nid, seed_+1); R = build(right_idx, depth+1, nid, seed_+2)
        nodes[nid].left, nodes[nid].right = L, R; return nid
    _ = build(np.arange(P_aug.shape[0]), 0, None, seed)
    # edges & structures
    edges = []
    for cid, nd in enumerate(nodes):
        if nd.parent is not None:
            p = nodes[nd.parent]; w = max(0.0, p.height - nd.height)
            edges.append((nd.parent, cid, w))
    leaf_index_map = {nid: i for i, nid in enumerate(leaf_ids)}
    E, L = len(edges), len(leaf_ids)
    S_edge_leaf = np.zeros((E, L), dtype=np.float32)
    def collect_leaves(nid, out):
        nd = nodes[nid]
        if nd.is_leaf: out.append(nid); return
        if nd.left is not None: collect_leaves(nd.left, out)
        if nd.right is not None: collect_leaves(nd.right, out)
    for e, (pid, cid, _) in enumerate(edges):
        leaves = []; collect_leaves(cid, leaves)
        for ln in leaves:
            j = leaf_index_map[ln]; S_edge_leaf[e, j] = 1.0
    centroids = np.vstack([P_aug[nd.idx].mean(axis=0) for nd in nodes])
    model = OTSWModel(
        P=P_aug, Sidx=Sidx, Tpos=Tpos, lengths=lengths, m_seq=m_seq, d=d,
        nodes=nodes, leaf_ids=leaf_ids, leaf_index_map=leaf_index_map,
        edges=edges, S_edge_leaf=S_edge_leaf, centroids=centroids,
        mode="tamle", lam_time=lam_time
    )
    _precompute_H_M(model)
    return model

# =========================
# 3) DISTANCE APIs
# =========================
def otsw_between_series_fast(model: OTSWModel, s_ref: int, s_cmp: int) -> float:
    """
    OTSW(s_ref, s_cmp) với cache:
      cost = sum_e w_e * |M[e, s_ref] - M[e, s_cmp]|
    """
    w = model.w; M = model.M
    diff = np.abs(M[:, s_ref] - M[:, s_cmp])
    return float((w * diff).sum())

def otsw_between_series(model: OTSWModel, s_ref: int, s_cmp: int, p: int = 1) -> float:
    assert p == 1, "Hiện tại hỗ trợ p=1 (W1    cây)."
    return otsw_between_series_fast(model, s_ref, s_cmp)

## 2. ASW Distance (Auto-weighted Sequential Wasserstein)
Implements ASW distance with GPU support via CuPy.

In [3]:
import numpy as np
import ot

try:
    import cupy as cp
except ImportError:
    cp = None


def asw_distance(
    A,
    B,
    lam=10.0,          # như TAOT: reg = 1 / lam cho Sinkhorn
    auto_weight=True,  # bật/tắt auto-weight cho 3 term
    w_spatial=1.0,     # weight cho spatial khi không auto
    w_order=1.0,       # weight cho order khi không auto
    w_struct=1.0,      # weight cho structural khi không auto
    tolerance=5e-3,
):
    """
    ASW distance (phiên bản gần đúng theo tinh thần Auto-weighted Sequential Wasserstein),
    hỗ trợ GPU nếu A hoặc B là cupy.ndarray.

    Ý tưởng:
    - C_spatial(i,j) = ||x_i - y_j||^2
    - C_order(i,j)   = (t_i - s_j)^2  với t_i, s_j là vị trí chuẩn hoá trong [0,1]
    - C_struct(i,j)  = ||gA_i - gB_j||^2, trong đó gA_i, gB_j là "gradient/cấu trúc lân cận"
      (chênh lệch giữa phần tử hiện tại và phần tử trước nó).

    - Tổng cost: C = w_s * C_spatial + w_o * C_order + w_n * C_struct
      với bộ weight có thể:
        + auto_weight=True: w_s, w_o, w_n lấy tự động từ dữ liệu (xấp xỉ ASW gốc),
        + auto_weight=False: dùng w_spatial, w_order, w_struct do người dùng cung cấp.

    Sau đó giải Sinkhorn như TAOT/POW:
      reg = 1 / lam
      asw = sum_{i,j} P_ij * C_ij
    """

    # 0) Backend: NumPy (CPU) hoặc CuPy (GPU)
    use_gpu = (cp is not None) and (
        isinstance(A, (cp.ndarray,)) or isinstance(B, (cp.ndarray,))
    )
    xp = cp if use_gpu else np

    # 1) Đưa dữ liệu về đúng backend, đảm bảo shape (n,d)
    A = xp.asarray(A, dtype=xp.float64)
    B = xp.asarray(B, dtype=xp.float64)

    if A.ndim == 1:
        A = A[:, None]
    if B.ndim == 1:
        B = B[:, None]

    n, dA = A.shape
    m, dB = B.shape
    if n == 0 or m == 0:
        raise ValueError("empty")
    if dA != dB:
        raise ValueError("dim mismatch")

    # ===== 2) C_spatial: khoảng cách đặc trưng =====
    diff = A[:, None, :] - B[None, :, :]
    C_spatial = xp.sum(diff * diff, axis=2)  # (n, m)

    # ===== 3) C_order: khoảng cách vị trí (index) =====
    # Chuẩn hoá index về [0,1]
    if n > 1:
        t = xp.linspace(0.0, 1.0, n)
    else:
        t = xp.zeros(1, dtype=xp.float64)

    if m > 1:
        s = xp.linspace(0.0, 1.0, m)
    else:
        s = xp.zeros(1, dtype=xp.float64)

    C_order = (t[:, None] - s[None, :]) ** 2  # (n, m)

    # ===== 4) C_struct: khoảng cách cấu trúc/gradient =====
    # Gradient đơn giản: gA[i] = A[i] - A[i-1], với gA[0] = 0
    gA = xp.zeros_like(A)
    if n > 1:
        gA[1:] = A[1:] - A[:-1]

    gB = xp.zeros_like(B)
    if m > 1:
        gB[1:] = B[1:] - B[:-1]

    gdiff = gA[:, None, :] - gB[None, :, :]
    C_struct = xp.sum(gdiff * gdiff, axis=2)  # (n, m)

    # ===== 5) Auto-weight hay dùng weight tay =====
    def _mean_safe(M):
        mu = xp.mean(M)
        if not xp.isfinite(mu) or float(mu) == 0.0:
            return 1.0
        return float(mu)

    if auto_weight:
        # Trọng số tỉ lệ nghịch với độ lớn trung bình của từng term:
        # term nào lớn quá -> weight nhỏ lại, để các term cân bằng hơn.
        mu_s = _mean_safe(C_spatial)
        mu_o = _mean_safe(C_order)
        mu_n = _mean_safe(C_struct)

        ws = 1.0 / mu_s
        wo = 1.0 / mu_o
        wn = 1.0 / mu_n
    else:
        ws = float(w_spatial)
        wo = float(w_order)
        wn = float(w_struct)

    C_base = ws * C_spatial + wo * C_order + wn * C_struct  # (n, m)

    # ===== 6) Chuẩn hoá cost cho Sinkhorn (giống TAOT/POW) =====
    med = xp.median(C_base)
    if not xp.isfinite(med) or float(med) <= 0.0:
        med = 1.0

    C = C_base / med

    # ===== 7) Khối lượng đều + Sinkhorn =====
    a = xp.full(n, 1.0 / n, dtype=xp.float64)
    b = xp.full(m, 1.0 / m, dtype=xp.float64)

    reg = 1.0 / lam

    P = ot.sinkhorn(a, b, C, reg, stopThr=tolerance)

    # ===== 8) ASW distance =====
    distance_raw = float(xp.sum(P * C_base))
    return distance_raw


## 3. TAOT Distance (Time-Adaptive Optimal Transport)
Implements TAOT distance with GPU support.

In [4]:
import numpy as np
import ot

try:
    import cupy as cp
except ImportError:
    cp = None


def taot_distance(A, B, lam=10.0, w=10.0, tolerance=5e-3):
    """
    TAOT distance, hỗ trợ GPU nếu A hoặc B là cupy.ndarray.

    - Nếu A/B là numpy array hoặc list -> chạy trên CPU (NumPy).
    - Nếu A hoặc B là cupy.ndarray -> convert cả hai sang CuPy và chạy Sinkhorn trên GPU.
    API (tham số/hàm trả về) giữ nguyên.
    """
    # Chọn backend: NumPy (CPU) hoặc CuPy (GPU)
    use_gpu = (cp is not None) and (
        isinstance(A, (cp.ndarray,)) or isinstance(B, (cp.ndarray,))
    )
    xp = cp if use_gpu else np

    # Đưa dữ liệu về đúng backend
    A = xp.asarray(A, dtype=xp.float64)
    B = xp.asarray(B, dtype=xp.float64)

    if A.ndim == 1:
        A = A[:, None]
    if B.ndim == 1:
        B = B[:, None]

    n, dA = A.shape
    m, dB = B.shape
    if n == 0 or m == 0:
        raise ValueError("empty")
    if dA != dB:
        raise ValueError("dim mismatch")

    # Chuẩn hoá chỉ số thời gian t,s (z-score) bằng backend xp
    t = xp.linspace(1, n, n)
    s = xp.linspace(1, m, m)

    def _zscore(x):
        mu = x.mean()
        std = x.std()
        if float(std) == 0.0:
            return x * 0.0
        return (x - mu) / std

    t = _zscore(t)
    s = _zscore(s)

    # Ma trận cost M (data + term bảo toàn thứ tự)
    diff = A[:, None, :] - B[None, :, :]
    M = xp.sum(diff * diff, axis=2) + w * (t[:, None] - s[None, :]) ** 2

    # Chuẩn hoá để đưa về C cho Sinkhorn
    med = xp.median(M)
    if not xp.isfinite(med) or float(med) <= 0.0:
        med = 1.0
    C = M / med

    # Khối lượng đều
    a = xp.full(n, 1.0 / n, dtype=xp.float64)
    b = xp.full(m, 1.0 / m, dtype=xp.float64)

    reg = 1.0 / lam

    # POT sẽ tự nhận backend dựa trên kiểu mảng (NumPy hoặc CuPy)
    P = ot.sinkhorn(a, b, C, reg, stopThr=tolerance)

    # Tính distance trong cùng backend rồi ép về float Python
    distance_raw = float(xp.sum(P * M))
    return distance_raw


## 4. POW Distance (Partial Ordered Wasserstein)
Implements POW distance with bandwidth constraint and GPU support.

In [5]:
import numpy as np
import ot

try:
    import cupy as cp
except ImportError:
    cp = None


def pow_distance(
    A,
    B,
    lam=10.0,          # giống TAOT: điều khiển reg = 1/lam cho Sinkhorn
    lam_order=1.0,     # hệ số cho regularization theo thứ tự (lambda_1)
    bandwidth=None,    # băng |i-j| cho phép match; nếu None sẽ auto
    tolerance=5e-3,
):
    """
    POW distance (phiên bản gần đúng theo tinh thần Partial Ordered Wasserstein),
    hỗ trợ GPU nếu A hoặc B là cupy.ndarray.

    Ý tưởng:
    - Ground cost: ||x_i - y_j||^2 + lam_order * |i/n - j/m|
    - Chỉ cho phép match trong một băng theo chỉ số |i - j| <= bandwidth
      (mô phỏng "partial" + hạn chế match xa).
    - Dùng Sinkhorn (entropic OT) như TAOT, reg = 1 / lam.

    Tham số:
      A, B      : (n,d), (m,d) NumPy hoặc CuPy array.
      lam       : tham số entropic (reg = 1/lam).
      lam_order : trọng số cho thành phần bảo toàn thứ tự.
      bandwidth : nếu None -> auto = 0.25 * max(n, m); nếu là số nguyên -> dùng trực tiếp.
      tolerance : ngưỡng dừng Sinkhorn.

    Trả về:
      distance_raw : scalar float (Python float).
    """
    # Chọn backend: NumPy (CPU) hoặc CuPy (GPU)
    use_gpu = (cp is not None) and (
        isinstance(A, (cp.ndarray,)) or isinstance(B, (cp.ndarray,))
    )
    xp = cp if use_gpu else np

    # Đưa dữ liệu về đúng backend
    A = xp.asarray(A, dtype=xp.float64)
    B = xp.asarray(B, dtype=xp.float64)

    # Đảm bảo dạng (n, d)
    if A.ndim == 1:
        A = A[:, None]
    if B.ndim == 1:
        B = B[:, None]

    n, dA = A.shape
    m, dB = B.shape
    if n == 0 or m == 0:
        raise ValueError("empty")
    if dA != dB:
        raise ValueError("dim mismatch")

    # 1) Cost đặc trưng: ||x_i - y_j||^2
    diff = A[:, None, :] - B[None, :, :]
    M_base = xp.sum(diff * diff, axis=2)  # shape (n, m)

    # 2) Regularization tuyến tính theo thứ tự: lam_order * |i/n - j/m|
    #    (khác với TAOT: dùng |.|, không bình phương)
    posA = xp.linspace(1, n, n) / float(n)   # 1/n, 2/n, ..., 1
    posB = xp.linspace(1, m, m) / float(m)   # 1/m, ..., 1
    order_term = xp.abs(posA[:, None] - posB[None, :])

    M_base = M_base + lam_order * order_term

    # 3) Bandwidth (partial constraint): chỉ cho phép match trong |i - j| <= bandwidth
    #    Nếu không cho, ta đặt chi phí Sinkhorn rất lớn ở ngoài băng.
    if bandwidth is None:
        # auto: 1/4 độ dài lớn hơn, làm int >= 1
        max_len = max(n, m)
        bandwidth = max(1, int(0.25 * max_len))

    idx_i = xp.arange(n)[:, None]
    idx_j = xp.arange(m)[None, :]
    mask = (xp.abs(idx_i - idx_j) <= bandwidth)  # True nếu được phép match

    # 4) Chuẩn hoá cost cho Sinkhorn
    med = xp.median(M_base)
    if not xp.isfinite(med) or float(med) <= 0.0:
        med = 1.0

    C = M_base / med

    # Thêm penalty rất lớn ở ngoài băng (chỉ trong C dùng cho Sinkhorn)
    # để gần như cấm mass đi ra ngoài vùng cho phép.
    big_C = 1e3
    C = C + (~mask).astype(xp.float64) * big_C

    # 5) Khối lượng đều
    a = xp.full(n, 1.0 / n, dtype=xp.float64)
    b = xp.full(m, 1.0 / m, dtype=xp.float64)

    reg = 1.0 / lam

    # 6) Sinkhorn (POT tự nhận backend từ kiểu mảng)
    P = ot.sinkhorn(a, b, C, reg, stopThr=tolerance)

    # 7) Tính distance dùng cost "thật" M_base (không cộng big_C)
    distance_raw = float(xp.sum(P * M_base))
    return distance_raw


## 5. TCOT Distance (Temporally Coupled Optimal Transport)
Implements TCOT distance with GPU support.

In [6]:
import numpy as np
import ot

try:
    import cupy as cp
except ImportError:
    cp = None


def tcot_distance_series(x, y, lambda_pos: float = 1.0, reg: float = 0.1, num_iter: int = 1000):
    """
    Tính khoảng cách Temporally Coupled Optimal Transport (TCOT) giữa hai chuỗi x, y.

    Hỗ trợ:
      - CPU: nếu x, y là numpy.ndarray (hoặc list)  -> dùng NumPy + ot.sinkhorn
      - GPU: nếu x hoặc y là cupy.ndarray          -> dùng CuPy + ot.gpu.sinkhorn

    Parameters
    ----------
    x, y : array-like hoặc np.ndarray / cp.ndarray
        - 1D: shape (n,)
        - 2D: shape (n, d)

    lambda_pos : float
        Hệ số phạt lệch thời gian.

    reg : float
        Entropic regularization cho Sinkhorn.

    num_iter : int
        Số vòng lặp tối đa cho Sinkhorn.

    Returns
    -------
    float
        Giá trị OT(C) với ground cost TCOT.
    """
    # Chọn backend: NumPy (CPU) hoặc CuPy (GPU)
    use_gpu = (cp is not None) and (
        isinstance(x, (cp.ndarray,)) or isinstance(y, (cp.ndarray,))
    )
    xp = cp if use_gpu else np

    # Đưa dữ liệu về đúng backend
    x = xp.asarray(x, dtype=xp.float64)
    y = xp.asarray(y, dtype=xp.float64)

    # Ép về dạng (n, d)
    if x.ndim == 1:
        x = x[:, None]
    if y.ndim == 1:
        y = y[:, None]

    n, dx = x.shape
    m, dy = y.shape
    if dx != dy:
        raise ValueError(f"Dimension mismatch: x dim={dx}, y dim={dy}")
    if n == 0 or m == 0:
        raise ValueError("Empty time series")

    # Cost đặc trưng: ||x_i - y_j||^2
    diff = x[:, None, :] - y[None, :, :]   # (n, m, d)
    C_feat = xp.sum(diff * diff, axis=2)   # (n, m)

    # Vị trí thời gian t, s chuẩn hóa [0,1]
    if n > 1:
        t = xp.linspace(0.0, 1.0, n)
    else:
        t = xp.zeros(1, dtype=xp.float64)

    if m > 1:
        s = xp.linspace(0.0, 1.0, m)
    else:
        s = xp.zeros(1, dtype=xp.float64)

    pos_diff = xp.abs(t[:, None] - s[None, :])  # (n, m)
    C_time = lambda_pos * (pos_diff ** 2)

    # Ground cost TCOT
    C_base = C_feat + C_time
    
    # Chuẩn hóa cost cho Sinkhorn (giống TAOT)
    med = xp.median(C_base)
    if not xp.isfinite(med) or float(med) <= 0.0:
        med = 1.0
    C = C_base / med

    # Trọng số đều
    a = xp.full(n, 1.0 / n, dtype=xp.float64)
    b = xp.full(m, 1.0 / m, dtype=xp.float64)

    # Thêm stopThr để tăng tốc (giống TAOT)
    G = ot.sinkhorn(a, b, C, reg, numItermax=num_iter, stopThr=5e-3)
    cost = float(xp.sum(G * C_base))  # Dùng C_base (chưa chuẩn hóa) để tính distance

    return cost

## 6. OPW Distance
Optimized implementation with GPU support (auto CPU/GPU).

In [7]:
import numpy as np
import ot

def _get_xp(use_gpu="auto"):
    if use_gpu is False:
        return np, False
    try:
        import cupy as cp
        # ensure CUDA is usable (otherwise fall back to numpy)
        try:
            _ = cp.cuda.runtime.getDeviceCount()
            return cp, True
        except Exception:
            if use_gpu is True:
                raise
            return np, False
    except Exception:
        if use_gpu is True:
            raise
        return np, False


def opw_distance_series(
    x, y,
    lambda1=1.0,
    lambda2=0.1,
    sigma=0.1,
    use_gpu="auto",
    max_iter=1000,
    tol=5e-3,
):
    xp, on_gpu = _get_xp(use_gpu)

    x = xp.asarray(x, dtype=xp.float64)
    y = xp.asarray(y, dtype=xp.float64)
    if x.ndim == 1: x = x[:, None]
    if y.ndim == 1: y = y[:, None]

    Nx, _ = x.shape
    Ny, _ = y.shape

    a = xp.full((Nx,), 1.0 / Nx, dtype=xp.float64)
    b = xp.full((Ny,), 1.0 / Ny, dtype=xp.float64)

    # Ground cost D_ij = ||x_i - y_j||^2
    x2 = xp.sum(x * x, axis=1)[:, None]
    y2 = xp.sum(y * y, axis=1)[None, :]
    D = x2 + y2 - 2.0 * (x @ y.T)
    D = xp.maximum(D, 0.0)

    # Time geometry
    i_norm = xp.arange(Nx, dtype=xp.float64)[:, None] / Nx
    j_norm = xp.arange(Ny, dtype=xp.float64)[None, :] / Ny
    diff = i_norm - j_norm

    S = diff**2 + 1.0
    
    # Total cost: D + lambda1*S
    M_base = D + lambda1 * S
    
    # Chuẩn hóa cost giống TAOT để tăng tốc Sinkhorn
    med = xp.median(M_base)
    if not xp.isfinite(med) or float(med) <= 0.0:
        med = 1.0
    M = M_base / med
    
    # Sinkhorn đơn giản (nhanh hơn bregman_log_projection_batch)
    T = ot.sinkhorn(a, b, M, reg=lambda2, numItermax=max_iter, stopThr=tol)

    dist = xp.sum(T * M_base)
    return float(dist.get()) if on_gpu else float(dist)

## 7. Dataset Loading Utilities
Functions to load Human Action datasets.

In [8]:
import os
from os.path import basename, join
import sys
import joblib
def load_human_action_dataset(data_dir, dataset_name):
    '''
    Loads train and test data from the folder in which
    the Human Actions dataset are stored.
    '''
    X_train = joblib.load(os.path.join(data_dir, dataset_name, "X_train.pkl"))
    y_train = joblib.load(os.path.join(data_dir, dataset_name, "y_train.pkl"))
    X_test = joblib.load(os.path.join(data_dir, dataset_name, "X_test.pkl"))
    y_test = joblib.load(os.path.join(data_dir, dataset_name, "y_test.pkl"))

    print("Successfully loaded dataset:", dataset_name)
    print("Size of train data:", len(y_train))
    print("Size of test data:", len(y_test))

    return X_train, y_train, X_test, y_test

## 8. Run KMedoids 

In [9]:
# run_kmedoids_all_methods_one_alg.py

import numpy as np
import time
import pandas as pd
from collections import Counter
from sklearn.metrics import normalized_mutual_info_score, accuracy_score
from sklearn_extra.cluster import KMedoids

# Assume these functions are defined in other modules:
# from xxx import opw_distance_series_gpu, tcot_distance_series, pow_distance
# from xxx import build_otsw_tamle, otsw_between_series_fast


# ---------- utils ----------
def as_ragged_list(X):
    if isinstance(X, np.ndarray) and X.ndim == 3 and X.dtype != object:
        return [X[i] for i in range(X.shape[0])]
    return [np.asarray(x, dtype=float) for x in X]


def zscore_per_series(X_list, eps: float = 1e-8):
    """Z-score normalize each series individually (shape-based)."""
    out = []
    for x in X_list:
        x = np.asarray(x, dtype=float)
        mu = x.mean(axis=0, keepdims=True)
        sigma = x.std(axis=0, keepdims=True)
        sigma[sigma < eps] = 1.0
        out.append((x - mu) / sigma)
    return out


def eval_scores_acc_nmi(y_true, y_pred):
    """Calculate Accuracy (after mapping clusters to labels) and NMI."""
    nmi = normalized_mutual_info_score(y_true, y_pred)

    mp = {}
    for c in np.unique(y_pred):
        true_labels_in_cluster = y_true[y_pred == c]
        if len(true_labels_in_cluster) > 0:
            most_common_class = Counter(true_labels_in_cluster).most_common(1)[0][0]
            mp[c] = most_common_class
        else:
            mp[c] = -1

    y_pred_as_class = np.array([mp[c] for c in y_pred])
    valid_mask = y_pred_as_class != -1
    acc = accuracy_score(y_true[valid_mask], y_pred_as_class[valid_mask])

    return acc, nmi

def dtw_distance_series(x, y): 
    from tslearn.metrics import dtw 
    return float(dtw(x, y)) 

# ---------- build_matrix for pairwise algorithms ----------
def build_matrix(alg, X_list, sakoe=None):
    """
    Build pairwise distance matrix for one algorithm:
      - alg == "DTW"
      - alg == "OPW"
      - alg == "TCOT"
      - alg == "POW"
    (OTSW uses its own function below, not this one.)
    """
    n = len(X_list)
    D = np.zeros((n, n), dtype=float)

    for i in range(n):
        for j in range(i + 1, n):
            if alg == "OPW":
                d = opw_distance_series(X_list[i], X_list[j])
            elif alg == "TAOT":
                d = taot_distance(X_list[i], X_list[j])
            elif alg == "DTW":
                d = dtw_distance_series(X_list[i], X_list[j])
            elif alg == "TCOT":
                d = tcot_distance_series(X_list[i], X_list[j])
            elif alg == "POW":
                d = pow_distance(X_list[i], X_list[j])
            elif alg == "ASW":
                d = asw_distance(X_list[i], X_list[j])
            else:
                raise ValueError(f"Unknown alg '{alg}' for build_matrix")

            D[i, j] = D[j, i] = float(d)

    return D


# ---------- OTSW ----------
def build_OTSW_matrix(
    X_list,
    lam_time=0.5,
    leaf_size=4,
    max_depth=8,
    seed=0,
    num_trees=5,
):
    """
    Build OTSW distance matrix on per-series normalized sequences.
    Uses ensemble of multiple trees (default 5) and averages their distances.

    Returns:
      - D: averaged distance matrix across num_trees
      - time_tree: total time to build all trees
    """
    X_norm = X_list  # already z-score per-series from outside
    n = len(X_norm)

    start_time_tree = time.time()
    D_sum = np.zeros((n, n), dtype=float)

    for t in range(num_trees):
        model = build_otsw_tamle(
            X_norm,
            lam_time=lam_time,
            leaf_size=leaf_size,
            max_depth=max_depth,
            seed=seed + t,  # Different seed for each tree
        )

        for i in range(n):
            for j in range(i + 1, n):
                dij = otsw_between_series_fast(model, i, j)
                D_sum[i, j] += dij
                D_sum[j, i] += dij

    time_tree = time.time() - start_time_tree
    D = D_sum / num_trees  # Average across all trees

    return D, time_tree


# ---------- run K-Medoids for 1 dataset + 1 algorithm ----------
def run_kmedoids_one_alg_on_dataset(
    dataset_name,
    alg,
    Xtr,
    ytr,
    Xte,
    yte,
    sakoe=5,
    lam_time=0.5,
    leaf_size=4,
    max_depth=8,
    seed=0,
):
    """
    Run once:
      - build distance matrix using 'alg'
      - run K-Medoids
      - return 1 dict of results (1 row) for this dataset
    """
    # Statistics
    n_train = len(Xtr)
    n_test = len(Xte)

    if Xtr.ndim == 3:
        n_features = Xtr.shape[2]
    elif Xtr.ndim == 2 and hasattr(Xtr[0], "ndim") and Xtr[0].ndim > 1:
        n_features = Xtr[0].shape[1]
    else:
        n_features = 1

    y = np.concatenate([np.asarray(ytr), np.asarray(yte)])
    _, y_true = np.unique(y, return_inverse=True)
    k = len(np.unique(y_true))

    print(f"\n=== Dataset {dataset_name} | ALG={alg} ===")
    print(f"| Train: {n_train} | Test: {n_test} | Total: {n_train + n_test}")
    print(f"| Features (Dimensions): {n_features} | Classes (Labels): {k}")
    print("=====================================")

    # Prepare data
    Xtr_list_raw = as_ragged_list(Xtr)
    Xte_list_raw = as_ragged_list(Xte)

    Xtr_list = zscore_per_series(Xtr_list_raw)
    Xte_list = zscore_per_series(Xte_list_raw)
    X_list = Xtr_list + Xte_list

    # Build distance
    print(f"  [{alg}] Building distance matrix ...")
    if alg == "OTSW":
        start_time_dist = time.time()
        D, time_tree = build_OTSW_matrix(
            X_list,
            lam_time=lam_time,
            leaf_size=leaf_size,
            max_depth=max_depth,
            seed=seed,
        )
        time_dist_total = time.time() - start_time_dist
    else:
        start_time_dist = time.time()
        D = build_matrix(alg, X_list, sakoe=sakoe)
        time_dist_total = time.time() - start_time_dist
        time_tree = np.nan

    # K-Medoids
    print(f"  [{alg}] Running K-Medoids ...")
    kmed = KMedoids(
        n_clusters=k,
        metric="precomputed",
        max_iter=100,
        random_state=seed,
    )
    start_clust = time.time()
    kmed.fit(D)
    time_clust = time.time() - start_clust

    acc, nmi = eval_scores_acc_nmi(y_true, kmed.labels_)

    print(f"    {alg}: ACC={acc:.4f}, NMI={nmi:.4f}")
    print(f"    {alg}: Time_dist={time_dist_total:.4f}s, Time_clust={time_clust:.4f}s")
    if alg == "OTSW":
        print(f"    {alg}: Time_tree={time_tree:.4f}s")

    # Package results as 1 row
    if alg == "DTW":
        params = {"sakoe": sakoe}
    elif alg == "OTSW":
        params = {
            "lam_time": lam_time,
            "leaf_size": leaf_size,
            "max_depth": max_depth,
            "seed": seed,
        }
    else:
        # OPW / TCOT / POW → add params here if needed
        params = {}

    row = {
        "Dataset": dataset_name,
        "Method": alg,
        "ACC": float(acc),
        "NMI": float(nmi),
        "Time_Dist_Total": float(time_dist_total),
        "Time_Clust": float(time_clust),
        "Time_Tree": float(time_tree),
        "Params": repr(params),
    }
    return row

## 9. Main Execution: K-Medoids Clustering

**CHANGE THE `ALG` VARIABLE BELOW TO SELECT WHICH METHOD TO RUN:**

Available options:
- `"OTSW"` - Ordered Tree Sliced Wasserstein
- `"OPW"` - Ordered Preserving Wasserstein(requires GPU)
- `"TCOT"` - Temporally Coupled Optimal Transport
- `"POW"` - Partial Ordered Wasserstein
- `"ASW"` - Auto-weighted Sequential Wasserstein
- `"TAOT"` - Time Adaptive Optimal Transport

The script will:
1. Load all UCR datasets specified in the list
2. Compute pairwise distance matrices using the selected method
3. Run K-Medoids clustering
4. Evaluate with Accuracy and NMI metrics
5. Save results to `kmedoids_results_{method}.csv`

## 8.5. Grid Search for OTSW lam_time Parameter

This section performs grid search to find the optimal `lam_time` parameter for OTSW.
The search range is from 0.5 to 20.0 with step size 0.5 (40 values total).

In [10]:
def grid_search_otsw_lamtime(
    dataset_name: str,
    Xtr: np.ndarray,
    ytr: np.ndarray,
    Xte: np.ndarray,
    yte: np.ndarray,
    lam_time_values: list,
    leaf_size: int = 4,
    max_depth: int = 8,
    seed: int = 0,
):
    """
    Grid search for OTSW's lam_time parameter.
    Returns the best lam_time and all results.
    """
    print(f"\n=== Grid Search OTSW lam_time for {dataset_name} ===")
    print(f"Testing {len(lam_time_values)} values: {lam_time_values}")
    
    results = []
    best_acc = -1
    best_lam_time = None
    
    for lam_time in lam_time_values:
        print(f"\n  Testing lam_time = {lam_time}...")
        try:
            row = run_kmedoids_one_alg_on_dataset(
                dataset_name=dataset_name,
                alg="OTSW",
                Xtr=Xtr,
                ytr=ytr,
                Xte=Xte,
                yte=yte,
                sakoe=5,
                lam_time=lam_time,
                leaf_size=leaf_size,
                max_depth=max_depth,
                seed=seed,
            )
            results.append(row)
            
            acc = row["ACC"]
            print(f"    lam_time={lam_time}: ACC={acc:.4f}, NMI={row['NMI']:.4f}")
            
            if acc > best_acc:
                best_acc = acc
                best_lam_time = lam_time
                print(f"    ✓ New best ACC: {best_acc:.4f} at lam_time={best_lam_time}")
                
        except Exception as e:
            print(f"    ✗ Error with lam_time={lam_time}: {e}")
    
    print(f"\n=== Grid Search Complete ===")
    print(f"Best lam_time: {best_lam_time}")
    print(f"Best ACC: {best_acc:.4f}")
    
    return best_lam_time, best_acc, results


# Example usage for grid search on a single dataset
def run_grid_search_example():
    """
    Example: Run grid search on BasicMotions dataset
    """
    from tslearn.datasets import UCR_UEA_datasets
    
    ucruea = UCR_UEA_datasets()
    dataset_name = "BasicMotions"
    
    print(f"Loading {dataset_name}...")
    Xtr, ytr, Xte, yte = ucruea.load_dataset(dataset_name)
    
    # Define grid search range: 0.5 to 20 with step 0.5
    # Fine-grained search: 40 values total
    lam_time_values = [i * 0.5 for i in range(1, 41)]  # 0.5, 1.0, 1.5, ..., 19.5, 20.0
    
    best_lam_time, best_acc, results = grid_search_otsw_lamtime(
        dataset_name=dataset_name,
        Xtr=Xtr,
        ytr=ytr,
        Xte=Xte,
        yte=yte,
        lam_time_values=lam_time_values,
        leaf_size=4,
        max_depth=8,
        seed=0,
    )
    
    # Save grid search results
    df_results = pd.DataFrame(results)
    output_file = f"grid_search_otsw_{dataset_name}.csv"
    df_results.to_csv(output_file, index=False, float_format="%.4f")
    print(f"\n✅ Grid search results saved to {output_file}")
    
    return best_lam_time, best_acc, df_results


# Uncomment to run grid search
# best_lam_time, best_acc, df_results = run_grid_search_example()
# print(f"\nOptimal lam_time: {best_lam_time}")
# print(f"Best Accuracy: {best_acc:.4f}")

In [11]:
def run_grid_search_all_datasets():
    """
    Run grid search for OTSW lam_time on all UCR datasets.
    This finds the optimal lam_time for each dataset separately.
    """
    from tslearn.datasets import UCR_UEA_datasets
    
    ucruea = UCR_UEA_datasets()
    
    # List of UCR datasets (same as main execution)
    datasets = [
        "ArrowHead",
        "BasicMotions",
        "BeetleFly",
        "CBF",
        "Chinatown",
        "CinCECGTorso",
        "DiatomSizeReduction",
        "GunPointAgeSpan",
        "GunPointMaleVersusFemale",
        "GunPointOldVersusYoung",
        "Ham",
        "InsectEPGRegularTrain",
        "ItalyPowerDemand",
        "Meat",
        "MelbournePedestrian",
        "MoteStrain",
        "OliveOil",
        "Plane",
        "SmoothSubspace",
    ]
    
    # Grid search range: 0.5 to 20 with step 0.5
    # Fine-grained search: 40 values total
    lam_time_values = [i * 0.5 for i in range(1, 41)]  # 0.5, 1.0, 1.5, ..., 19.5, 20.0
    
    BASE_SEED = 0
    LEAF_SIZE = 4
    MAX_DEPTH = 8
    
    all_best_params = []
    all_grid_results = []
    
    print(f"--- Grid Search OTSW lam_time for {len(datasets)} UCR datasets ---")
    print(f"Testing lam_time values: {lam_time_values}\n")
    
    for dataset_name in datasets:
        try:
            print(f"\n{'='*60}")
            print(f"Dataset: {dataset_name}")
            print(f"{'='*60}")
            
            Xtr, ytr, Xte, yte = ucruea.load_dataset(dataset_name)
            
            # Downsample large datasets
            if dataset_name in ["CinCECGTorso", "MixedShapesSmallTrain"]:
                X_all = np.concatenate([Xtr, Xte], axis=0)
                y_all = np.concatenate([ytr, yte], axis=0)
                
                if len(X_all) > 300:
                    rng = np.random.default_rng(BASE_SEED)
                    idx = rng.choice(len(X_all), size=300, replace=False)
                    X_all = X_all[idx]
                    y_all = y_all[idx]
                    print(f"[DOWNSAMPLE] {dataset_name}: {len(Xtr)+len(Xte)} → 300 samples")
                
                n_train = int(0.7 * len(X_all))
                Xtr, ytr = X_all[:n_train], y_all[:n_train]
                Xte, yte = X_all[n_train:], y_all[n_train:]
            
            # Run grid search
            best_lam_time, best_acc, results = grid_search_otsw_lamtime(
                dataset_name=dataset_name,
                Xtr=Xtr,
                ytr=ytr,
                Xte=Xte,
                yte=yte,
                lam_time_values=lam_time_values,
                leaf_size=LEAF_SIZE,
                max_depth=MAX_DEPTH,
                seed=BASE_SEED,
            )
            
            # Store best parameters for this dataset
            all_best_params.append({
                "Dataset": dataset_name,
                "Best_lam_time": best_lam_time,
                "Best_ACC": best_acc,
            })
            
            # Store all grid results
            all_grid_results.extend(results)
            
            # Save intermediate results
            df_best = pd.DataFrame(all_best_params)
            df_best.to_csv("otsw_best_lamtime_per_dataset.csv", index=False, float_format="%.4f")
            
            df_all = pd.DataFrame(all_grid_results)
            df_all.to_csv("otsw_grid_search_all_results.csv", index=False, float_format="%.4f")
            
            print(f"\n✅ Progress saved. Completed {len(all_best_params)}/{len(datasets)} datasets")
            
        except Exception as e:
            print(f"\n!! ERROR processing {dataset_name}: {e}")
            print("   Skipping this dataset.")
    
    # Final summary
    if all_best_params:
        df_best = pd.DataFrame(all_best_params)
        print(f"\n{'='*60}")
        print("GRID SEARCH COMPLETE - BEST LAM_TIME PER DATASET")
        print(f"{'='*60}")
        print(df_best.to_string(index=False))
        
        avg_best_lam_time = df_best["Best_lam_time"].mean()
        median_best_lam_time = df_best["Best_lam_time"].median()
        
        print(f"\n--- Summary Statistics ---")
        print(f"Average best lam_time: {avg_best_lam_time:.2f}")
        print(f"Median best lam_time: {median_best_lam_time:.2f}")
        print(f"Average best ACC: {df_best['Best_ACC'].mean():.4f}")
        
        print(f"\n✅ Results saved to:")
        print(f"   - otsw_best_lamtime_per_dataset.csv (best params per dataset)")
        print(f"   - otsw_grid_search_all_results.csv (all grid search results)")
        
        return df_best, pd.DataFrame(all_grid_results)
    else:
        print("\n⚠️ No dataset completed successfully.")
        return None, None


# Uncomment to run grid search on all datasets
# df_best, df_all = run_grid_search_all_datasets()

## 8.6. How to Use Grid Search Results

After running the grid search, you can use the optimal `lam_time` values in several ways:

**Option 1: Use a single optimal value for all datasets**
- Calculate the average or median from `otsw_best_lamtime_per_dataset.csv`
- Set `LAM_TIME` in the main execution cell to this value

**Option 2: Use dataset-specific optimal values**
- Load the best parameters from the CSV file
- Create a dictionary mapping dataset names to their optimal `lam_time`
- Modify the main execution loop to use the dataset-specific value

**Option 3: Quick test with grid search example**
```python
# Uncomment and run to test on a single dataset
# best_lam_time, best_acc, df_results = run_grid_search_example()
```

**Option 4: Run full grid search on all datasets**
```python
# Uncomment and run to grid search all datasets (this will take time!)
# df_best, df_all = run_grid_search_all_datasets()
```

The grid search will:
1. Test lam_time values: 0.5, 1.0, 1.5, 2.0, ..., 19.5, 20.0 (40 values with step 0.5)
2. Save results to CSV files after each dataset
3. Report the best `lam_time` and accuracy for each dataset
4. Provide summary statistics across all datasets

In [12]:
# ---------- main ----------
if __name__ == "__main__":
    from tslearn.datasets import UCR_UEA_datasets

    # ONLY CHANGE THIS VARIABLE TO RUN A DIFFERENT METHOD
    # ALG = "DTW" / "OPW" / "TCOT" / "POW" / "OTSW"
    ALG = "OPW"

    ucruea = UCR_UEA_datasets()

    output_filename = f"kmedoids_results_{ALG.lower()}.csv"
    all_results = []

    # List of UCR datasets
    datasets = [
        "ArrowHead",              # AH
        "BasicMotions",           # BM
        "BeetleFly",              # BF
        "CBF",                    # CBF
        "Chinatown",              # CT
        "CinCECGTorso",           # CET
        "DiatomSizeReduction",    # DSR
        "GunPointAgeSpan",        # GPA
        "GunPointMaleVersusFemale", # GPM
        "GunPointOldVersusYoung", # GPO
        "Ham",                    # Ham
        "InsectEPGRegularTrain",  # IERT
        "ItalyPowerDemand",       # IPD
        "Meat",                   # Meat
        "MelbournePedestrian",    # MP
        "MixedShapesSmallTrain",  # MS2T
        "MoteStrain",             # MS
        "OliveOil",               # O2
        "Plane",                  # Plane
        "SmoothSubspace",         # S2
    ]
    ucr_uea_datasets = [
        "ArrowHead",              # AH
        "BasicMotions",           # BM
        "BeetleFly",              # BF
        "CBF",                    # CBF
        "Chinatown",              # CT
        "CinCECGTorso",           # CET
        "DiatomSizeReduction",    # DSR
        "GunPointAgeSpan",        # GPA
        "GunPointMaleVersusFemale", # GPM
        "GunPointOldVersusYoung", # GPO
        "Ham",                    # Ham
        "InsectEPGRegularTrain",  # IERT
        "ItalyPowerDemand",       # IPD
        "Meat",                   # Meat
        "MelbournePedestrian",    # MP
        "MoteStrain",             # MS
        "OliveOil",               # O2
        "Plane",                  # Plane
        "SmoothSubspace",         # S2
    ]

    SAKOE  = 5
    LEAF_SIZE = 4
    MAX_DEPTH = 8
    BASE_SEED = 0
    
    # OTSW lam_time parameter (temporal regularization weight)
    # Default: 20 (original value)
    # For optimal results, run grid search first (see section 8.5) to find best value per dataset
    # Grid search range: 0.5 to 20
    LAM_TIME = 20

    print(f"--- Running K-Medoids ({ALG}) for {len(ucr_uea_datasets)} UCR datasets ---")

    for dataset_name in ucr_uea_datasets:
        try:
            Xtr, ytr, Xte, yte = ucruea.load_dataset(dataset_name)
            
            # Downsample CinCECGTorso và MixedShapesSmallTrain xuống 300 mẫu
            if dataset_name in ["CinCECGTorso", "MixedShapesSmallTrain"]:
                X_all = np.concatenate([Xtr, Xte], axis=0)
                y_all = np.concatenate([ytr, yte], axis=0)
                
                if len(X_all) > 300:
                    rng = np.random.default_rng(BASE_SEED)
                    idx = rng.choice(len(X_all), size=300, replace=False)
                    X_all = X_all[idx]
                    y_all = y_all[idx]
                    print(f"[DOWNSAMPLE] {dataset_name}: {len(Xtr)+len(Xte)} → 300 samples")
                
                # Chia lại train/test 70/30
                n_train = int(0.7 * len(X_all))
                Xtr, ytr = X_all[:n_train], y_all[:n_train]
                Xte, yte = X_all[n_train:], y_all[n_train:]

            row = run_kmedoids_one_alg_on_dataset(
                dataset_name=dataset_name,
                alg=ALG,
                Xtr=Xtr,
                ytr=ytr,
                Xte=Xte,
                yte=yte,
                sakoe=SAKOE,
                lam_time=LAM_TIME,  # Use optimal value from grid search if available
                leaf_size=LEAF_SIZE,
                max_depth=MAX_DEPTH,
                seed=BASE_SEED,
            )

            all_results.append(row)
            df = pd.DataFrame(all_results)
            print(df.tail(1))  # print the row just completed

            df.to_csv(
                output_filename,
                index=False,
                float_format="%.4f",
            )
            print(f"✅ Temporarily saved results to {output_filename}")

        except Exception as e:
            print(f"!! ERROR processing {dataset_name}: {e}")
            print("   Skipping this dataset.")

    if all_results:
        print(f"\n✅ Completed {len(all_results)} rows (1 algorithm × {len(all_results)} datasets) → {output_filename}")
    else:
        print("\n⚠️ No dataset completed successfully.")

--- Running K-Medoids (OPW) for 19 UCR datasets ---

=== Dataset ArrowHead | ALG=OPW ===
| Train: 36 | Test: 175 | Total: 211
| Features (Dimensions): 1 | Classes (Labels): 3
  [OPW] Building distance matrix ...


KeyboardInterrupt: 

# 9. Run K-Medoids on Human Action Datasets

Run K-Medoids clustering on MSRAction3D, Weizmann, and SpokenArabicDigit datasets.
- SpokenArabicDigit (SAD) will be downsampled to 10% of its size

In [ ]:
# ---------- Run K-Medoids on Human Action Datasets ----------
if __name__ == "__main__":
    import joblib
    
    # ONLY CHANGE THIS VARIABLE TO RUN A DIFFERENT METHOD
    # ALG = "DTW" / "OPW" / "TCOT" / "POW" / "OTSW"
    ALG = "DTW"
    
    output_filename = f"kmedoids_results_human_actions_{ALG.lower()}.csv"
    all_results = []
    
    # List of Human Action datasets
    allhuman_action_datasets = [
        "MSRAction3D",
        "Weizmann",
        "SpokenArabicDigit",
    ]
    human_action_datasets = [
        "MSRAction3D"
    ]
    
    SAKOE = 5
    LEAF_SIZE = 12
    MAX_DEPTH = 20
    BASE_SEED = 0
    LAM_TIME = 10
    
    print(f"--- Running K-Medoids ({ALG}) for {len(human_action_datasets)} Human Action datasets ---")
    
    for dataset_name in human_action_datasets:
        try:
            # Load dataset from joblib pickle files
            data_path = f"../data/Human_Actions/{dataset_name}"
            
            print(f"\nLoading {dataset_name} from {data_path}...")
            
            Xtr = joblib.load(f"{data_path}/X_train.pkl")
            ytr = joblib.load(f"{data_path}/y_train.pkl")
            Xte = joblib.load(f"{data_path}/X_test.pkl")
            yte = joblib.load(f"{data_path}/y_test.pkl")
            
            # Convert to numpy arrays if they are lists
            if isinstance(Xtr, list):
                Xtr = np.array(Xtr, dtype=object)
            if isinstance(ytr, list):
                ytr = np.array(ytr)
            if isinstance(Xte, list):
                Xte = np.array(Xte, dtype=object)
            if isinstance(yte, list):
                yte = np.array(yte)
            
            print(f"  Train: {len(Xtr)} samples, Test: {len(Xte)} samples")
            
            # Downsample SpokenArabicDigit to 10%
            if dataset_name == "SpokenArabicDigit":
                X_all = np.concatenate([Xtr, Xte], axis=0)
                y_all = np.concatenate([ytr, yte], axis=0)
                
                target_size = int(len(X_all) * 0.1)
                rng = np.random.default_rng(BASE_SEED)
                idx = rng.choice(len(X_all), size=target_size, replace=False)
                X_all = X_all[idx]
                y_all = y_all[idx]
                
                print(f"  [DOWNSAMPLE] {dataset_name}: {len(Xtr)+len(Xte)} → {target_size} samples (10%)")
                
                # Split into train/test 70/30
                n_train = int(0.7 * len(X_all))
                Xtr, ytr = X_all[:n_train], y_all[:n_train]
                Xte, yte = X_all[n_train:], y_all[n_train:]
            
            row = run_kmedoids_one_alg_on_dataset(
                dataset_name=dataset_name,
                alg=ALG,
                Xtr=Xtr,
                ytr=ytr,
                Xte=Xte,
                yte=yte,
                sakoe=SAKOE,
                lam_time=LAM_TIME,
                leaf_size=LEAF_SIZE,
                max_depth=MAX_DEPTH,
                seed=BASE_SEED,
            )
            
            all_results.append(row)
            df = pd.DataFrame(all_results)
            print(df.tail(1))  # print the row just completed
            
            df.to_csv(
                output_filename,
                index=False,
                float_format="%.4f",
            )
            print(f"✅ Temporarily saved results to {output_filename}")
            
        except Exception as e:
            print(f"!! ERROR processing {dataset_name}: {e}")
            print("   Skipping this dataset.")
            import traceback
            traceback.print_exc()
    
    if all_results:
        print(f"\n✅ Completed {len(all_results)} rows (1 algorithm × {len(all_results)} datasets) → {output_filename}")
    else:
        print("\n⚠️ No dataset completed successfully.")

## 10. Ablation Study for OTSW Parameters

This section performs ablation study on OTSW hyperparameters:
- **Lambda (lam_time)**: (0.001, 0.005, 0.01, 0.05, 0.1, 0.5, 1, 5, 10, 50, 100)
- **Max Depth**: (5, 10, 15, 20, 25, 30)
- **Number of Trees**: (1, 3, 5, 7, 9, 11, 13, 15)
- **Number of Clusters (k_split)**: (2, 4, 8, 16, 32)

**Default values**: lambda=5, depth=30, trees=5, num_cluster=2

When varying one parameter, all others are fixed at default values.
Results include NMI, Accuracy, and execution time for each configuration.

In [ ]:
# ============================================================
# ABLATION STUDY FOR OTSW PARAMETERS
# ============================================================

import numpy as np
import pandas as pd
import time
import matplotlib.pyplot as plt
from tslearn.datasets import UCR_UEA_datasets

# ---------------------- Configuration ----------------------
# Default parameter values
DEFAULT_LAMBDA = 5
DEFAULT_DEPTH = 30
DEFAULT_TREES = 5
DEFAULT_NUM_CLUSTER = 2  # k_split

# Parameter ranges for ablation
LAMBDA_VALUES = [0.001, 0.005, 0.01, 0.05, 0.1, 0.5, 1, 5, 10, 50, 100]
DEPTH_VALUES = [5, 10, 15, 20, 25, 30]
TREES_VALUES = [1, 3, 5, 7, 9, 11, 13, 15]
NUM_CLUSTER_VALUES = [2, 4, 8, 16, 32]

# Dataset to use for ablation study
ABLATION_DATASET = "BasicMotions"  # Change this to test on different datasets
LEAF_SIZE = 4
BASE_SEED = 0

# ---------------------- Modified build_OTSW_matrix with k_split ----------------------
def build_OTSW_matrix_ablation(
    X_list,
    lam_time=5.0,
    leaf_size=4,
    max_depth=30,
    seed=0,
    num_trees=5,
    k_split=2,
):
    """
    Build OTSW distance matrix with configurable k_split for ablation study.
    """
    X_norm = X_list  # already z-score per-series from outside
    n = len(X_norm)

    start_time_tree = time.time()
    D_sum = np.zeros((n, n), dtype=float)

    for t in range(num_trees):
        model = build_otsw_tamle(
            X_norm,
            lam_time=lam_time,
            leaf_size=leaf_size,
            max_depth=max_depth,
            seed=seed + t,
            k_split=k_split,
        )

        for i in range(n):
            for j in range(i + 1, n):
                dij = otsw_between_series_fast(model, i, j)
                D_sum[i, j] += dij
                D_sum[j, i] += dij

    time_tree = time.time() - start_time_tree
    D = D_sum / num_trees  # Average across all trees

    return D, time_tree

N_RUNS = 5

# ---------------------- Single ablation run ----------------------
def run_ablation_single(
    X_list, y_true, k,
    lam_time=DEFAULT_LAMBDA,
    max_depth=DEFAULT_DEPTH,
    num_trees=DEFAULT_TREES,
    k_split=DEFAULT_NUM_CLUSTER,
    seed=BASE_SEED,
):
    """
    Run K-Medoids with OTSW for a single parameter configuration.
    Returns: (ACC, NMI, time_total)
    """
    start_time = time.time()
    
    D, _ = build_OTSW_matrix_ablation(
        X_list,
        lam_time=lam_time,
        leaf_size=LEAF_SIZE,
        max_depth=max_depth,
        seed=seed,
        num_trees=num_trees,
        k_split=k_split,
    )
    
    kmed = KMedoids(
        n_clusters=k,
        metric="precomputed",
        max_iter=100,
        random_state=seed,
    )
    kmed.fit(D)
    
    acc, nmi = eval_scores_acc_nmi(y_true, kmed.labels_)
    time_total = time.time() - start_time
    
    return acc, nmi, time_total


# ---------------------- Run ablation for one parameter ----------------------
def run_ablation_for_param(X_list, y_true, k, param_name, param_values, n_runs=N_RUNS, **fixed_params):
    """
    Run ablation study for a single parameter with multiple runs.
    Returns: DataFrame with columns [param_value, ACC_mean, ACC_std, NMI_mean, NMI_std, Time_mean, Time_std]
    """
    results = []
    
    print(f"\n{'='*60}")
    print(f"Ablation Study: {param_name}")
    print(f"Testing {len(param_values)} values: {param_values}")
    print(f"Number of runs per value: {n_runs}")
    print(f"Fixed params: {fixed_params}")
    print(f"{'='*60}")
    
    for val in param_values:
        params = fixed_params.copy()
        params[param_name] = val
        
        print(f"  Testing {param_name}={val}...")
        
        acc_list = []
        nmi_list = []
        time_list = []
        
        # Run multiple times with different seeds
        for run_idx in range(n_runs):
            try:
                acc, nmi, time_total = run_ablation_single(
                    X_list, y_true, k, 
                    seed=BASE_SEED + run_idx,
                    **params
                )
                acc_list.append(acc)
                nmi_list.append(nmi)
                time_list.append(time_total)
                print(f"    Run {run_idx+1}/{n_runs}: ACC={acc:.4f}, NMI={nmi:.4f}, Time={time_total:.2f}s")
            except Exception as e:
                print(f"    Run {run_idx+1}/{n_runs}: ERROR - {e}")
                acc_list.append(np.nan)
                nmi_list.append(np.nan)
                time_list.append(np.nan)
        
        # Calculate mean and std
        acc_mean = np.nanmean(acc_list)
        acc_std = np.nanstd(acc_list)
        nmi_mean = np.nanmean(nmi_list)
        nmi_std = np.nanstd(nmi_list)
        time_mean = np.nanmean(time_list)
        time_std = np.nanstd(time_list)
        
        print(f"  Summary: ACC={acc_mean:.4f}±{acc_std:.4f}, NMI={nmi_mean:.4f}±{nmi_std:.4f}, Time={time_mean:.2f}±{time_std:.2f}s\n")
        
        results.append({
            param_name: val,
            "ACC_mean": acc_mean,
            "ACC_std": acc_std,
            "NMI_mean": nmi_mean,
            "NMI_std": nmi_std,
            "Time_mean": time_mean,
            "Time_std": time_std,
        })
    
    return pd.DataFrame(results)


# ---------------------- Plotting function with mean and std ----------------------
def plot_ablation_results(df, param_name, output_dir="ablation_results"):
    """
    Plot ablation results with mean and std: ACC, NMI, and Time vs parameter value.
    Uses shaded regions to show standard deviation.
    """
    import os
    os.makedirs(output_dir, exist_ok=True)
    
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    
    x = df[param_name].astype(str)
    x_numeric = range(len(x))
    
    # Plot ACC with std
    acc_mean = df["ACC_mean"]
    acc_std = df["ACC_std"]
    axes[0].plot(x_numeric, acc_mean, marker='o', linewidth=2, markersize=8, color='blue', label='Mean')
    axes[0].fill_between(x_numeric, acc_mean - acc_std, acc_mean + acc_std, alpha=0.2, color='blue')
    axes[0].errorbar(x_numeric, acc_mean, yerr=acc_std, fmt='none', ecolor='blue', alpha=0.5, capsize=3)
    axes[0].set_xlabel(param_name, fontsize=12)
    axes[0].set_ylabel("Accuracy", fontsize=12)
    axes[0].set_title(f"Accuracy vs {param_name}", fontsize=14)
    axes[0].set_xticks(x_numeric)
    axes[0].set_xticklabels(x, rotation=45, ha='right')
    axes[0].grid(True, alpha=0.3)
    axes[0].legend()
    
    # Plot NMI with std
    nmi_mean = df["NMI_mean"]
    nmi_std = df["NMI_std"]
    axes[1].plot(x_numeric, nmi_mean, marker='s', linewidth=2, markersize=8, color='green', label='Mean')
    axes[1].fill_between(x_numeric, nmi_mean - nmi_std, nmi_mean + nmi_std, alpha=0.2, color='green')
    axes[1].errorbar(x_numeric, nmi_mean, yerr=nmi_std, fmt='none', ecolor='green', alpha=0.5, capsize=3)
    axes[1].set_xlabel(param_name, fontsize=12)
    axes[1].set_ylabel("NMI", fontsize=12)
    axes[1].set_title(f"NMI vs {param_name}", fontsize=14)
    axes[1].set_xticks(x_numeric)
    axes[1].set_xticklabels(x, rotation=45, ha='right')
    axes[1].grid(True, alpha=0.3)
    axes[1].legend()
    
    # Plot Time with std
    time_mean = df["Time_mean"]
    time_std = df["Time_std"]
    axes[2].plot(x_numeric, time_mean, marker='^', linewidth=2, markersize=8, color='red', label='Mean')
    axes[2].fill_between(x_numeric, time_mean - time_std, time_mean + time_std, alpha=0.2, color='red')
    axes[2].errorbar(x_numeric, time_mean, yerr=time_std, fmt='none', ecolor='red', alpha=0.5, capsize=3)
    axes[2].set_xlabel(param_name, fontsize=12)
    axes[2].set_ylabel("Time (seconds)", fontsize=12)
    axes[2].set_title(f"Execution Time vs {param_name}", fontsize=14)
    axes[2].set_xticks(x_numeric)
    axes[2].set_xticklabels(x, rotation=45, ha='right')
    axes[2].grid(True, alpha=0.3)
    axes[2].legend()
    
    plt.tight_layout()
    
    # Save figure
    fig_path = os.path.join(output_dir, f"ablation_{param_name}.png")
    plt.savefig(fig_path, dpi=150, bbox_inches='tight')
    plt.show()
    
    print(f"✅ Plot saved to {fig_path}")
    return fig_path


# ---------------------- Main ablation study ----------------------
def run_full_ablation_study(dataset_name=ABLATION_DATASET):
    """
    Run complete ablation study for all OTSW parameters.
    """
    print(f"\n{'#'*70}")
    print(f"# OTSW ABLATION STUDY ON DATASET: {dataset_name}")
    print(f"{'#'*70}")
    
    # Load dataset
    ucruea = UCR_UEA_datasets()
    Xtr, ytr, Xte, yte = ucruea.load_dataset(dataset_name)
    
    # Prepare data
    Xtr_list = zscore_per_series(as_ragged_list(Xtr))
    Xte_list = zscore_per_series(as_ragged_list(Xte))
    X_list = Xtr_list + Xte_list
    
    y = np.concatenate([np.asarray(ytr), np.asarray(yte)])
    _, y_true = np.unique(y, return_inverse=True)
    k = len(np.unique(y_true))
    
    print(f"\nDataset: {dataset_name}")
    print(f"Total samples: {len(X_list)}")
    print(f"Number of classes: {k}")
    print(f"\nDefault parameters:")
    print(f"  - Lambda (lam_time): {DEFAULT_LAMBDA}")
    print(f"  - Max Depth: {DEFAULT_DEPTH}")
    print(f"  - Number of Trees: {DEFAULT_TREES}")
    print(f"  - Number of Clusters (k_split): {DEFAULT_NUM_CLUSTER}")
    
    all_results = {}
    '''
    # 1. Ablation on Lambda (lam_time)
    print("\n" + "="*70)
    print("1. ABLATION ON LAMBDA (lam_time)")
    print("="*70)
    df_lambda = run_ablation_for_param(
        X_list, y_true, k,
        param_name="lam_time",
        param_values=LAMBDA_VALUES,
        max_depth=DEFAULT_DEPTH,
        num_trees=DEFAULT_TREES,
        k_split=DEFAULT_NUM_CLUSTER,
    )
    df_lambda.to_csv("ablation_lambda.csv", index=False)
    plot_ablation_results(df_lambda, "lam_time")
    all_results["lambda"] = df_lambda
    
    # 2. Ablation on Max Depth
    print("\n" + "="*70)
    print("2. ABLATION ON MAX DEPTH")
    print("="*70)
    df_depth = run_ablation_for_param(
        X_list, y_true, k,
        param_name="max_depth",
        param_values=DEPTH_VALUES,
        lam_time=DEFAULT_LAMBDA,
        num_trees=DEFAULT_TREES,
        k_split=DEFAULT_NUM_CLUSTER,
    )
    df_depth.to_csv("ablation_depth.csv", index=False)
    plot_ablation_results(df_depth, "max_depth")
    all_results["depth"] = df_depth
    '''
    # 3. Ablation on Number of Trees
    print("\n" + "="*70)
    print("3. ABLATION ON NUMBER OF TREES")
    print("="*70)
    df_trees = run_ablation_for_param(
        X_list, y_true, k,
        param_name="num_trees",
        param_values=TREES_VALUES,
        lam_time=DEFAULT_LAMBDA,
        max_depth=DEFAULT_DEPTH,
        k_split=DEFAULT_NUM_CLUSTER,
    )
    df_trees.to_csv("ablation_trees.csv", index=False)
    plot_ablation_results(df_trees, "num_trees")
    all_results["trees"] = df_trees
    
    # 4. Ablation on Number of Clusters (k_split)
    print("\n" + "="*70)
    print("4. ABLATION ON NUMBER OF CLUSTERS (k_split)")
    print("="*70)
    df_cluster = run_ablation_for_param(
        X_list, y_true, k,
        param_name="k_split",
        param_values=NUM_CLUSTER_VALUES,
        lam_time=DEFAULT_LAMBDA,
        max_depth=DEFAULT_DEPTH,
        num_trees=DEFAULT_TREES,
    )
    df_cluster.to_csv("ablation_cluster.csv", index=False)
    plot_ablation_results(df_cluster, "k_split")
    all_results["cluster"] = df_cluster
    
    # Summary
    print("\n" + "#"*70)
    print("# ABLATION STUDY COMPLETE")
    print("#"*70)
    print("\nResults saved to ablation_results/ folder:")
    print("  - ablation_lambda.csv + ablation_lam_time.png")
    print("  - ablation_depth.csv + ablation_max_depth.png")
    print("  - ablation_trees.csv + ablation_num_trees.png")
    print("  - ablation_cluster.csv + ablation_k_split.png")
    
    return all_results


# ---------------------- Run the ablation study ----------------------
# Uncomment the following line to run the full ablation study:
all_results = run_full_ablation_study(dataset_name="CBF")

# Or run individual parameter ablations:
# Example: Run only lambda ablation
# df_lambda = run_ablation_for_param(X_list, y_true, k, "lam_time", LAMBDA_VALUES, max_depth=30, num_trees=5, k_split=2)